In [24]:
import numpy as np
import pandas as pd
import re   # searching from keywords
from nltk.corpus import stopwords  # articles [a, an, the,in, on, at, with, to, of,he, she, it, they, I, me, my ]
from nltk.stem.porter import PorterStemmer # [ use for root word for perticular work]
from sklearn.feature_extraction.text import TfidfVectorizer # covert in text in feature vectores
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression # for geting fake [0] or not fack [1] binary pridction
from sklearn.metrics import accuracy_score

In [25]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [26]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [27]:
# data collection and data preprocessing

import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohamadalhasan/a-fake-news-dataset-around-the-syrian-war")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'a-fake-news-dataset-around-the-syrian-war' dataset.
Path to dataset files: /kaggle/input/a-fake-news-dataset-around-the-syrian-war


In [32]:
import pandas as pd

news_dataset = pd.read_csv(
    '/root/.cache/kagglehub/datasets/mohamadalhasan/a-fake-news-dataset-around-the-syrian-war/versions/1/FA-KES-Dataset.csv',
    encoding='latin1'
)

print(news_dataset.head())

      unit_id                                      article_title  \
0  1914947530  Syria attack symptoms consistent with nerve ag...   
1  1914947532  Homs governor says U.S. attack caused deaths b...   
2  1914947533    Death toll from Aleppo bomb attack at least 112   
3  1914947534        Aleppo bomb blast kills six Syrian state TV   
4  1914947535  29 Syria Rebels Dead in Fighting for Key Alepp...   

                                     article_content source       date  \
0  Wed 05 Apr 2017 Syria attack symptoms consiste...    nna   4/5/2017   
1  Fri 07 Apr 2017 at 0914 Homs governor says U.S...    nna   4/7/2017   
2  Sun 16 Apr 2017 Death toll from Aleppo bomb at...    nna  4/16/2017   
3  Wed 19 Apr 2017 Aleppo bomb blast kills six Sy...    nna  4/19/2017   
4  Sun 10 Jul 2016 29 Syria Rebels Dead in Fighti...    nna  7/10/2016   

  location  labels  
0    idlib       0  
1     homs       0  
2   aleppo       0  
3   aleppo       0  
4   aleppo       0  


,unit_id,article_title,article_content,date,location,labels
source,,,,,,
ahram,286653385898,10 dead after strikes on rebel-held hospitals ...,10 dead after strikes on rebel-held hospitals ...,4/27/20174/19/20174/16/20174/16/20174/15/20174...,idlibaleppoaleppoaleppoaleppoaleppodamascusraq...,74
alalam,176624611613,Bomb blast kills 60 in Damascus suburbCar bomb...,Fri Sep 27 2013 Bomb blast kills 60 in Damascu...,9/27/20132/2/20118/31/20133/15/20128/1/20138/2...,damascusdamascusdamascuslattakiahomsraqqadeir ...,49
alaraby,161448613484,Syrian regime steps up aerial assault on Douma...,12 February 2015 Casualties mount in the Easte...,2/12/20152/12/20152/23/20157/6/20157/7/20157/8...,damascusdamascusidlibraqqaaleppoaleppoidlibale...,48
arabiya,42382542995,Family of six die in east Aleppo chlorine gas ...,Tuesday 22 November 2016 A photo apparently sh...,11/22/201610/7/20175/18/20178/25/20134/6/20171...,aleppoaleppohamadamascusidlibaleppodamascusdam...,10
asharqalawsat,21099650374,Chemical Massacre in Idlib Defies the WorldBus...,April 5 2017 by Youssef Diab Paula Astih and N...,4/5/20174/18/20174/19/20174/4/20173/25/20178/3...,idlibaleppoaleppoidlibhamaidlibaleppoaleppodam...,7
dailysabah,142301073184,Syrian army advances near AleppoMonitor 10 of ...,Published February 18 2015 The Syrian army bac...,2/18/20152/23/20152/23/20157/5/20157/20/20157/...,aleppoaleppolattakiaraqqalattakiaaleppoidlibid...,40
etilaf,88618038589,135 Civilians Killed in Escalating Bombardment...,Wednesday 07 December 2016 135 Civilians Kille...,12/7/20164/23/20167/22/20157/31/201411/21/2014...,idlibdamascusaleppoaleppoaleppodeir ezzoridlib...,20
jordantimes,53873625469,Israeli attack on Syria military camp kills th...,Last updated at Apr 232017 An Israeli attack o...,4/13/20174/13/20174/14/20174/15/20178/19/20168...,quneitraalepporaqqaidlibaleppoaleppoidlibalepp...,20
manar,206144479385,16 Civilians Dead in Coalition Strikes near Ra...,May 24 2017 16 Civilians Dead in Coalition Str...,5/24/20175/30/20175/18/20174/19/20174/10/20174...,raqqadeir ezzorhamaaleppoidlibidlibhomsidlibda...,54


In [34]:
# here 1 - real news and 0 - fake news

In [35]:
# check the number of missing values in the dataset

news_dataset.isnull().sum()

,0
unit_id,0
article_title,0
article_content,0
source,0
date,0
location,0
labels,0


In [135]:
# merging the author name and title

news_dataset['content'] =  news_dataset['source']+' : ' + news_dataset['article_title']

In [136]:
print(news_dataset['content'])

0      nna : Syria attack symptoms consistent with ne...
1      nna : Homs governor says U.S. attack caused de...
2      nna : Death toll from Aleppo bomb attack at le...
3      nna : Aleppo bomb blast kills six Syrian state TV
4      nna : 29 Syria Rebels Dead in Fighting for Key...
                             ...                        
799    manar : Turkish Bombardment Kills 20 Civilians...
800    manar : Martyrs as Terrorists Shell Aleppos Sa...
801    manar : Chemical Attack Kills Five Syrians in ...
802    manar : 5 Killed as Russian Military Chopper S...
803    manar : Syrian Army Kills 48 ISIL Terrorists i...
Name: content, Length: 804, dtype: object


In [137]:
x = news_dataset.drop(columns='labels', axis=1)
y = news_dataset['labels']

In [138]:
print(x)
print(y)

        unit_id                                      article_title  \
0    1914947530  Syria attack symptoms consistent with nerve ag...   
1    1914947532  Homs governor says U.S. attack caused deaths b...   
2    1914947533    Death toll from Aleppo bomb attack at least 112   
3    1914947534        Aleppo bomb blast kills six Syrian state TV   
4    1914947535  29 Syria Rebels Dead in Fighting for Key Alepp...   
..          ...                                                ...   
799  1965511221    Turkish Bombardment Kills 20 Civilians in Syria   
800  1965511222    Martyrs as Terrorists Shell Aleppos Salah Eddin   
801  1965511224  Chemical Attack Kills Five Syrians in Aleppo SANA   
802  1965511226  5 Killed as Russian Military Chopper Shot down...   
803  1965511231  Syrian Army Kills 48 ISIL Terrorists in Deir E...   

                                       article_content source       date  \
0    Wed 05 Apr 2017 Syria attack symptoms consiste...    nna   4/5/2017   
1    Fr

stemming : stemming is process of reducing a word to it's root word

 ex. actor, actoress, acting --> act

In [139]:
port_stem = PorterStemmer()

In [140]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ',content) # a set where small and capital [ A-Z ]
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content


In [141]:
news_dataset['content'] = news_dataset['content'].apply(stemming)

In [142]:
news_dataset['content']

,content
0,nna syria attack symptom consist nerv agent use
1,nna hom governor say u attack caus death doesn...
2,nna death toll aleppo bomb attack least
3,nna aleppo bomb blast kill six syrian state tv
4,nna syria rebel dead fight key aleppo road
...,...
799,manar turkish bombard kill civilian syria
800,manar martyr terrorist shell aleppo salah eddin
801,manar chemic attack kill five syrian aleppo sana
802,manar kill russian militari chopper shot syria


In [143]:
x = news_dataset['content'].values
y = news_dataset['labels'].values

In [144]:
print(x)

['nna syria attack symptom consist nerv agent use'
 'nna hom governor say u attack caus death doesnt see big human loss'
 'nna death toll aleppo bomb attack least'
 'nna aleppo bomb blast kill six syrian state tv'
 'nna syria rebel dead fight key aleppo road'
 'nna suicid bomb kill least northeast syria'
 'nna dead heavi u raid syria stronghold'
 'nna suicid bomber kill assad clan hometown'
 'nna explos rock town damascu' 'nna damascu explos due rocket bomb'
 'alarabi syrian regim step aerial assault douma'
 'alarabi hizballah lead regim offens southern syria'
 'alarabi syrian opposit remain divid'
 'alarabi video show murder syrian activist'
 'alarabi syria nusra front stage deadli suicid bomb aleppo'
 'alarabi regim troop thwart rebel attack syria aleppo'
 'alarabi ahrar al sham leader kill syria'
 'alarabi barrel bomb kill town syria'
 'alarabi rebel advanc north western syria'
 'alarabi isra strike syrian town kill pro regim fighter'
 'alarabi syria armi plane crash rebel held town

In [145]:
print(y)

[0 0 0 0 0 0 0 0 1 0 1 1 1 1 0 0 1 1 0 0 0 0 1 1 1 1 1 0 0 1 0 0 1 0 0 1 1
 1 1 1 1 1 1 0 1 1 1 1 1 1 0 0 1 1 0 0 1 0 1 1 1 1 0 1 1 1 0 0 0 1 0 1 0 0
 1 0 1 1 0 0 0 0 0 0 1 1 1 1 0 0 0 0 1 1 0 1 1 1 1 1 0 1 0 0 1 1 1 0 0 1 1
 1 1 1 1 1 0 1 1 1 1 1 1 1 1 0 1 0 0 1 0 1 0 1 0 1 1 1 1 1 0 1 1 1 0 1 0 0
 1 1 0 0 0 1 1 1 1 0 1 1 1 1 1 0 0 1 1 0 1 0 1 0 0 0 0 1 1 0 0 1 0 1 1 0 0
 1 0 1 0 0 0 0 1 0 0 1 1 0 0 0 0 0 1 0 0 0 1 0 1 1 1 1 1 0 0 0 1 1 0 1 0 1
 0 0 0 0 0 1 0 0 0 0 1 1 0 1 1 1 0 1 1 0 0 0 1 1 0 1 1 0 0 1 1 0 1 1 0 1 1
 0 1 1 0 0 1 1 1 1 0 0 0 0 1 1 0 1 0 0 0 1 0 0 1 0 0 1 1 1 1 1 1 1 1 1 1 0
 1 0 0 0 1 1 1 0 1 1 1 1 1 0 0 1 0 1 1 0 1 0 0 0 1 0 1 1 1 1 1 0 0 0 0 1 1
 0 1 1 1 0 0 1 0 0 1 1 1 1 0 1 1 0 0 0 1 0 0 0 0 0 1 0 1 0 1 0 0 1 0 0 0 1
 1 0 1 0 1 1 0 0 0 0 1 0 1 0 1 1 1 1 1 0 1 1 1 1 0 1 0 0 1 0 0 0 0 1 1 0 1
 1 1 0 1 1 0 0 1 1 0 0 0 1 1 1 0 0 0 1 1 1 0 0 0 0 1 1 1 1 1 0 0 1 1 0 1 0
 0 0 0 1 1 0 0 0 0 1 0 1 0 0 0 0 0 1 0 0 0 0 1 1 1 1 0 0 1 0 0 1 1 1 1 1 0
 0 1 0 0 1 1 0 0 0 1 0 1 

In [146]:
# coverting sentences to meaning full numbers - here we use vectors

vectorizer = TfidfVectorizer()  # convert raw text documents into a matrix of numerical features.
vectorizer.fit(x)

x = vectorizer.transform(x)



In [147]:
print(x)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7116 stored elements and shape (804, 775)>
  Coords	Values
  (0, 21)	0.4398385037739128
  (0, 71)	0.154270763028172
  (0, 156)	0.4398385037739128
  (0, 465)	0.4143530883600743
  (0, 471)	0.31752877227058074
  (0, 674)	0.4398385037739128
  (0, 675)	0.1123974588039333
  (0, 737)	0.33268695746858934
  (1, 71)	0.12184316409612725
  (1, 95)	0.34738477945645097
  (1, 122)	0.3272563792892771
  (1, 188)	0.24114602484679362
  (1, 211)	0.34738477945645097
  (1, 301)	0.30189759411838357
  (1, 324)	0.20673629748416938
  (1, 334)	0.34738477945645097
  (1, 407)	0.30189759411838357
  (1, 471)	0.2507844619783276
  (1, 603)	0.22200068141769194
  (1, 612)	0.34738477945645097
  (2, 36)	0.21999407466746793
  (2, 71)	0.23968501507821843
  (2, 99)	0.2929163001914418
  (2, 188)	0.47437284668556384
  (2, 394)	0.3389939588680877
  :	:
  (800, 622)	0.34302181140418664
  (800, 692)	0.2062183140289721
  (801, 36)	0.24495050157124273
  (801, 71)	0.26687

In [148]:
# here now we fit this data to machine learning model

# here split our x and y

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.1, stratify=y, random_state=2)

# stratify = While splitting, keep the same proportion of classes.
# random_state = Shuffle in a special fixed way so I get the same result every time.
# Today's split → Different
# Tomorrow's split → Different


In [149]:
model = LogisticRegression()

In [150]:
# training the model : Logistic Regression model [getting binory ]

model.fit(x_train, y_train)

LogisticRegression()

In [151]:
# accuracy score

x_train_prediction = model.predict(x_train)
training_data_accuracy = accuracy_score(x_train_prediction, y_train)

In [152]:
print('the accuracy of :',training_data_accuracy*100)

the accuracy of : 78.42323651452283


In [153]:
x_test_prediction = model.predict(x_test)
testing_data_accuracy = accuracy_score(x_test_prediction, y_test)

In [154]:
print('the accuracy of :',testing_data_accuracy*100)

the accuracy of : 58.0246913580247


In [155]:
# making predictive system

x_new = x_test[0]

prediction = model.predict(x_new)

print(prediction)

if(prediction[0]==0):
  print('the news is real')
else:
  print('the news is fake')

[1]
the news is fake


In [158]:
news_dataset['source'].unique()

array(['nna', 'alaraby', 'asharqalawsat', 'dailysabah', 'trt', 'ahram',
       'jordantimes', 'tass', 'sana', 'etilaf', 'manar', 'arabiya',
       'reuters', 'alalam', 'sputnik'], dtype=object)

In [132]:
# Install widgets (run once)
 !pip install -q ipywidgets


In [156]:
# ===============================
# GUI FOR FAKE NEWS DETECTION
# ===============================



import ipywidgets as widgets
from IPython.display import display, HTML

# -------------------------------
# Prediction Function
# -------------------------------

def predict_news(source, headline):

    # Same format used during training
    content = source + " : " + headline

    # Preprocess
    processed = stemming(content)

    # Convert to TF-IDF
    vector = vectorizer.transform([processed])

    # Prediction
    prediction = model.predict(vector)

    return prediction[0]


# -------------------------------
# Source Input
# -------------------------------

source_box = widgets.Text(
    placeholder='Example: ndtv',
    description='Source:',
    layout=widgets.Layout(width='600px')
)

# -------------------------------
# Headline Input
# -------------------------------

headline_box = widgets.Textarea(
    placeholder='Paste news headline here...',
    description='Headline:',
    layout=widgets.Layout(width='800px', height='120px')
)

# -------------------------------
# Predict Button
# -------------------------------

predict_button = widgets.Button(
    description='Check News',
    button_style='success',
    icon='search'
)

# -------------------------------
# Output Area
# -------------------------------

output = widgets.Output()


# -------------------------------
# Button Function
# -------------------------------

def check_news(button):

    output.clear_output()

    source = source_box.value.strip()
    headline = headline_box.value.strip()

    with output:

        if source == "" or headline == "":
            display(HTML(
                "<h3 style='color:red;'>Please enter Source and Headline.</h3>"
            ))
            return


        result = predict_news(source, headline)


        if result == 0:
            display(HTML(
                """
                <h2 style='color:green;'>
                🟢 REAL NEWS
                </h2>
                """
            ))
        else:
            display(HTML(
                """
                <h2 style='color:red;'>
                🔴 FAKE NEWS
                </h2>
                """
            ))


# Connect button
predict_button.on_click(check_news)


# -------------------------------
# Display GUI
# -------------------------------

display(HTML("<h2>Fake News Detection System</h2>"))

display(source_box)
display(headline_box)
display(predict_button)
display(output)

Text(value='', description='Source:', layout=Layout(width='600px'), placeholder='Example: ndtv')

Textarea(value='', description='Headline:', layout=Layout(height='120px', width='800px'), placeholder='Paste n…

Button(button_style='success', description='Check News', icon='search', style=ButtonStyle())

Output()